# Phase 1.3 — Data Cleaning: Unit Shift, Numerical Types & Confounder Prep

**Owner:** Rahma Bahgat (Person B — ID: 20231056)  
**Module:** `src/unit_shift_cleaning.py`  
**Tests:** `tests/test_unit_shift_cleaning.py`  
**Branch:** `phase1/Pb-unit-shift-cleaning`

---

## Objective

Translate **Person B's Phase 1 Audit Findings (Task 1b: Unit Shift & Task 1d: Confounding)** into a reproducible, train-only calibrated cleaning module for the team pipeline (`trips.csv`, `recent_labelled.csv`, `holdout_inputs.csv`, and `predict.py`):

1. **Unit Shift Repair (`est_fare` piasters → EGP):**
   - Historical `trips.csv` (May 1 – Aug 30, 2026) records `est_fare` in EGP (median `90.10`, median `fare/km = 9.33`).
   - Starting **2026-08-31** (`recent_labelled.csv` & `holdout_inputs.csv`), an app release switched `est_fare` to piasters (`×100`; median `9,090–9,190`, `fare/km >= 482.27`).
   - Normalize shifted fares back to EGP (`/ 100.0`) using a **train-only** calibrated threshold (`idempotent`, zero leakage).
2. **Data Types & Temporal Confounder Extraction:**
   - Parse `request_ts` to `datetime64[ns]`, validate numerical dtypes (`float64`), and extract `hour`, `day_of_week`, `month` (controls for `hour` confounding `surge_multiplier`).
3. **Zone Confounder Context & Simpson's Paradox Verification:**
   - Attach `pickup_zone_type` and `pickup_has_metro` from `zones.csv` and verify Simpson's Paradox on completed trips (`global_slope = -0.1856` vs `within_slope = +0.5067` min/km).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.unit_shift_cleaning import (
    UnitShiftAndConfounderCleaner,
    compute_confounding_slopes,
    detect_unit_shift_onset,
)

for folder_name in ("cairogo_pack", "CairoGo_pack_v17"):
    candidate = ROOT / "data" / "raw" / folder_name
    if (candidate / "trips.csv").exists():
        PACK = candidate
        break
else:
    raise FileNotFoundError("Could not locate Pack v17 CSV directory under data/raw/")

print(f"Using pack directory: {PACK.name}")

Using pack directory: cairogo_pack


In [2]:
trips_raw   = pd.read_csv(PACK / "trips.csv")
recent_raw  = pd.read_csv(PACK / "recent_labelled.csv")
holdout_raw = pd.read_csv(PACK / "holdout_inputs.csv")
zones_df    = pd.read_csv(PACK / "zones.csv")

print(f"Loaded trips   : {trips_raw.shape}")
print(f"Loaded recent  : {recent_raw.shape}")
print(f"Loaded holdout : {holdout_raw.shape}")
print(f"Loaded zones   : {zones_df.shape}")

print("\n=== EST_FARE MEDIANS BEFORE CLEANING ===")
for name, df in [("trips", trips_raw), ("recent", recent_raw), ("holdout", holdout_raw)]:
    fpk_med = (df["est_fare"] / df["distance_km"]).median()
    print(f"{name:8s} -> est_fare median = {df['est_fare'].median():8.2f}, fare/km median = {fpk_med:6.2f}")

Loaded trips   : (28244, 18)
Loaded recent  : (3533, 18)
Loaded holdout : (2767, 13)
Loaded zones   : (24, 5)

=== EST_FARE MEDIANS BEFORE CLEANING ===
trips    -> est_fare median =    90.10, fare/km median =   9.33
recent   -> est_fare median =  9090.00, fare/km median = 925.84
holdout  -> est_fare median =  9190.00, fare/km median = 938.46


In [3]:
# Task 1b Audit Check: Detect shifted field & exact onset day
shift_audit = detect_unit_shift_onset(trips_raw, recent_raw)
print("=== UNIT SHIFT ONSET DETECTION (TASK 1b) ===")
for k, v in shift_audit.items():
    print(f"  {k:30s}: {v}")

# Task 1d Audit Check: Verify Simpson's Paradox confounding on completed trips
confound_audit = compute_confounding_slopes(trips_raw, zones_df)
print("\n=== CONFOUNDING SLOPES (TASK 1d: actual_trip_time_min ~ distance_km) ===")
for k, v in confound_audit.items():
    print(f"  {k:22s}: {v}")

=== UNIT SHIFT ONSET DETECTION (TASK 1b) ===
  field                         : est_fare
  day_estimate                  : 2026-08-31
  scale_factor_estimate         : 99.17
  baseline_fare_per_km_median   : 9.3294
  shifted_fare_per_km_median    : 925.1841
  shifted_days_count            : 14



=== CONFOUNDING SLOPES (TASK 1d: actual_trip_time_min ~ distance_km) ===
  confounder            : pickup_zone_type
  n_completed_trips     : 23221
  global_slope          : -0.1856
  within_slope          : 0.5067
  group_slopes          : {'airport': 0.5962, 'downtown': 0.5609, 'midtown': 0.5092, 'new_city': 0.4302, 'suburb': 0.421}


In [4]:
# Fit strictly on training data (trips.csv) — zero leakage from validation or holdout
cleaner = UnitShiftAndConfounderCleaner().fit(trips_raw)

print("=== LEARNED TRAIN-ONLY PARAMETERS ===")
for k, v in cleaner.stats_.items():
    print(f"  {k:28s}: {v}")

trips_clean = cleaner.transform(trips_raw, zones_df=zones_df)
trips_stats = cleaner.last_transform_stats_

recent_clean = cleaner.transform(recent_raw, zones_df=zones_df)
recent_stats = cleaner.last_transform_stats_

holdout_clean = cleaner.transform(holdout_raw, zones_df=zones_df)
holdout_stats = cleaner.last_transform_stats_

print("\n=== UNIT CORRECTION SUMMARY ===")
for name, stats, df_c in [
    ("trips (Train)", trips_stats, trips_clean),
    ("recent (Val)", recent_stats, recent_clean),
    ("holdout (Test)", holdout_stats, holdout_clean),
]:
    fpk_after = (df_c["est_fare"] / df_c["distance_km"]).median()
    print(
        f"{name:15s} -> corrected={stats['rows_unit_corrected']:4d}/{stats['rows_in']:5d} "
        f"({stats['share_unit_corrected']*100:5.1f}%), est_fare median={df_c['est_fare'].median():6.2f} EGP, "
        f"fare/km median={fpk_after:5.2f}"
    )

=== LEARNED TRAIN-ONLY PARAMETERS ===
  train_rows                  : 28244
  train_fare_per_km_median    : 9.3294
  train_fare_per_km_p99       : 35.7985
  train_fare_per_km_max       : 77.4349
  fare_per_km_threshold       : 186.5875
  fallback_fare_threshold     : 1500.0
  scale_factor                : 100.0
  unit_shift_field            : est_fare
  unit_shift_date             : 2026-08-31



=== UNIT CORRECTION SUMMARY ===
trips (Train)   -> corrected=   0/28244 (  0.0%), est_fare median= 90.10 EGP, fare/km median= 9.33
recent (Val)    -> corrected=3533/ 3533 (100.0%), est_fare median= 90.90 EGP, fare/km median= 9.26
holdout (Test)  -> corrected=2767/ 2767 (100.0%), est_fare median= 91.90 EGP, fare/km median= 9.38


In [5]:
# Integrity, Immutability & Idempotency Assertions
for name, raw_df, clean_df in [
    ("trips", trips_raw, trips_clean),
    ("recent", recent_raw, recent_clean),
    ("holdout", holdout_raw, holdout_clean),
]:
    assert len(clean_df) == len(raw_df), f"Row count mismatch in {name}"
    assert pd.api.types.is_datetime64_any_dtype(clean_df["request_ts"]), f"request_ts not datetime in {name}"
    for col in ("hour", "day_of_week", "month", "pickup_zone_type", "pickup_has_metro"):
        assert col in clean_df.columns, f"Missing {col} in {name}"
        assert clean_df[col].isna().sum() == 0, f"Unexpected NaNs in {name}.{col}"

# Idempotency check on holdout
holdout_recleaned = cleaner.transform(holdout_clean, zones_df=zones_df)
assert cleaner.last_transform_stats_["rows_unit_corrected"] == 0, "Expected 0 changes on second pass"
pd.testing.assert_frame_equal(holdout_clean, holdout_recleaned)

print("All Person B cleaning assertions passed (immutability, scale alignment, temporal/zone confounders, idempotency).")

All Person B cleaning assertions passed (immutability, scale alignment, temporal/zone confounders, idempotency).
